In [1]:
pip install folium pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install folium pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
%%html
<div id="map" style="width: 100%; height: 500px; background: #f0f0f0; border-radius: 8px;"></div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>

<script>
    // 1. Initialisation de la carte
    var map = L.map('map').setView([46.603354, 1.888334], 6);

    // 2. Ajout du fond de carte
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{z}.png', {
        maxZoom: 19,
        attribution: '&copy; CARTO'
    }).addTo(map);

    // 3. Chargement des données des régions françaises
    fetch('https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions.geojson')
        .then(response => response.json())
        .then(data => {
            var geojsonLayer = L.geoJson(data, {
                style: {
                    fillColor: '#3186cc',
                    weight: 1.5,
                    opacity: 1,
                    color: 'white',
                    fillOpacity: 0.4
                },
                onEachFeature: function(feature, layer) {
                    var nomRegion = feature.properties.nom;
                    layer.bindTooltip("<b>" + nomRegion + "</b>", { sticky: true });
                    
                    layer.on({
                        mouseover: function(e) {
                            e.target.setStyle({
                                fillColor: '#ff7800',
                                fillOpacity: 0.7
                            });
                        },
                        mouseout: function(e) {
                            geojsonLayer.resetStyle(e.target);
                        }
                    });
                }
            }).addTo(map);
        })
        .catch(err => console.error("Erreur de chargement :", err));

    // 4. Forcer le redimensionnement pour éviter l'affichage vide dans Jupyter
    setTimeout(function() {
        map.invalidateSize();
    }, 300);
</script>

In [ ]:
import os  # Pour gérer les chemins de fichiers et vérifier s'ils existent
import zipfile  # Pour créer une archive compressée (.zip) de la carte HTML
import folium  # Pour créer la carte interactive
from folium.plugins import MarkerCluster  # Pour regrouper les antennes en clusters (plus fluide et lisible)
import pandas as pd  # Pour charger et nettoyer les données tabulaires


def charger_donnees(chemin_fichier, limite=None):
  """Charge le fichier et convertit les coordonnées DMS en degrés décimaux."""
  print("Chargement et nettoyage des données...")  # Message de suivi dans la console
  df = pd.read_csv(chemin_fichier, sep=";", low_memory=False)  # Lit le fichier séparé par des points-virgules

  # --- Conversion de la latitude (Degrés, Minutes, Secondes -> degrés décimaux) ---
  deg_lat = pd.to_numeric(df["COR_NB_DG_LAT"], errors="coerce")  # Convertit les degrés en nombres (NaN si invalide)
  min_lat = pd.to_numeric(df["COR_NB_MN_LAT"], errors="coerce")  # Convertit les minutes en nombres
  sec_lat = pd.to_numeric(df["COR_NB_SC_LAT"], errors="coerce")  # Convertit les secondes en nombres
  lat = deg_lat + min_lat / 60.0 + sec_lat / 3600.0  # Formule de conversion DMS vers décimal
  lat = lat.where(df["COR_CD_NS_LAT"] != "S", -lat).round(5)  # Rend la latitude négative si au Sud ('S') et arrondit à 5 décimales

  # --- Conversion de la longitude (Degrés, Minutes, Secondes -> degrés décimaux) ---
  deg_lon = pd.to_numeric(df["COR_NB_DG_LON"], errors="coerce")  # Convertit les degrés de longitude en nombres
  min_lon = pd.to_numeric(df["COR_NB_MN_LON"], errors="coerce")  # Convertit les minutes de longitude en nombres
  sec_lon = pd.to_numeric(df["COR_NB_SC_LON"], errors="coerce")  # Convertit les secondes de longitude en nombres
  lon = deg_lon + min_lon / 60.0 + sec_lon / 3600.0  # Formule de conversion DMS vers décimal
  lon = lon.where(df["COR_CD_EW_LON"] != "W", -lon).round(5)  # Rend la longitude négative si à l'Ouest ('W') et arrondit à 5 décimales

  df["LAT"] = lat  # Ajoute la colonne latitude décimale au tableau
  df["LON"] = lon  # Ajoute la colonne longitude décimale au tableau
  df = df.dropna(subset=["LAT", "LON"])  # Supprime les lignes qui n'ont pas de coordonnées valides

  if limite is not None:  # Si une limite est précisée (ex: 10000)
    df = df.head(limite)  # Garde seulement les 'limite' premières lignes

  return df  # Renvoie le tableau nettoyé


def classifier_antenne(val):
  """Retourne (couleur, groupe_filtre, libelle_detaille) selon le code TPO_ID."""
  if pd.isna(val):  # Si le code TPO_ID est vide / manquant
    return "#111111", "multi", "Multi / TPO non renseigné"  # Couleur noire, groupe 'multi', libellé explicite
  try:
    tpo = int(float(val))  # Convertit par exemple "74.0" ou 74.0 en entier 74
  except Exception:
    return "#111111", "multi", f"Multi / TPO ({val})"  # Si la conversion échoue, classe en Multi / Autre

  if tpo == 74:  # Code associé à Orange
    return "#ff8c00", "orange", "Orange (TPO 74)"  # Orange
  elif tpo == 32:  # Code associé à SFR
    return "#e11d48", "sfr", "SFR (TPO 32)"  # Rouge
  elif tpo == 29:  # Code associé à Free Mobile
    return "#7c2d12", "free", "Free Mobile (TPO 29)"  # Marron
  elif tpo == 72:  # Code associé à Bouygues Telecom
    return "#2563eb", "bouygues", "Bouygues Telecom (TPO 72)"  # Bleu
  else:  # Tous les autres codes TPO_ID
    return "#111111", "multi", f"Multi / Autre (TPO {tpo})"  # Noir avec le numéro TPO affiché


def creer_carte(df):
  """Crée une carte libre de droits (OpenStreetMap / OpenTopoMap) avec

  antennes bien visibles, regroupées, filtrables et légendées.
  """
  carte = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles=None)  # Crée la carte centrée sur la France (zoom 6), sans fond par défaut

  # 1. OpenStreetMap (ODbL - libre de droits, sans clé API)
  folium.TileLayer(
      tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",  # URL des tuiles OpenStreetMap
      attr="&copy; OpenStreetMap contributors",  # Mention obligatoire de la source
      name="OpenStreetMap (ODbL)",  # Nom affiché dans le sélecteur de couches
      user_agent="SAE15_Antennes_France/1.0",  # Identifiant de requête pour éviter certains blocages 403
      max_zoom=19,  # Niveau de zoom maximum autorisé
  ).add_to(carte)  # Ajoute cette couche à la carte

  # 2. OpenTopoMap (CC-BY-SA - libre de droits, sans clé API)
  folium.TileLayer(
      tiles="https://tile.opentopomap.org/{z}/{x}/{y}.png",  # URL des tuiles topographiques OpenTopoMap
      attr=(
          "&copy; OpenStreetMap contributors, SRTM | Map style: &copy;"
          " OpenTopoMap (CC-BY-SA)"
      ),  # Mention de licence CC-BY-SA
      name="OpenTopoMap (CC-BY-SA)",  # Nom dans le sélecteur de couches
      max_zoom=17,  # Zoom maximum pour OpenTopoMap
  ).add_to(carte)  # Ajoute cette seconde couche à la carte

  # Liste des grandes villes françaises pour repère visuel (nom, latitude, longitude)
  villes = [
      ("Paris", 48.8566, 2.3522),
      ("Lyon", 45.7640, 4.8357),
      ("Marseille", 43.2965, 5.3698),
      ("Toulouse", 43.6047, 1.4442),
      ("Bordeaux", 44.8378, -0.5792),
      ("Lille", 50.6292, 3.0573),
      ("Nantes", 47.2184, -1.5536),
      ("Strasbourg", 48.5734, 7.7521),
  ]
  for nom, lat, lon in villes:  # Parcourt chaque ville
    folium.Marker(
        location=[lat, lon],  # Position de la ville
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:11px;font-weight:bold;color:#222;'
                f"background:rgba(255,255,255,0.88);padding:2px 5px;"
                f'border-radius:3px;border:1px solid #888;">{nom}</div>'
            )  # Petite étiquette texte lisible sur la carte
        ),
    ).add_to(carte)  # Ajoute l'étiquette à la carte

  # Configuration des groupes d'opérateurs pour créer un cluster par catégorie
  groupes_config = {
      "bouygues": ("Bouygues Telecom", "#2563eb"),
      "orange": ("Orange", "#ff8c00"),
      "sfr": ("SFR", "#e11d48"),
      "free": ("Free Mobile", "#7c2d12"),
      "multi": ("Multi / Autre", "#111111"),
  }

  clusters_par_groupe = {}  # Dictionnaire pour stocker un MarkerCluster par catégorie
  for cle, (nom_grp, _) in groupes_config.items():  # Pour chaque opérateur / catégorie
    mc = MarkerCluster(name=nom_grp, show=True)  # Crée un cluster visible par défaut
    mc.add_to(carte)  # Ajoute le cluster à la carte
    clusters_par_groupe[cle] = mc  # Enregistre le cluster sous sa clé

  folium.LayerControl(position="bottomright", collapsed=False).add_to(carte)  # Ajoute le panneau de couches/filtres en bas à droite, déplié

  print(f"Ajout de {len(df)} antennes sur la carte...")  # Affiche le nombre total d'antennes traitées
  for row in df.itertuples(index=False):  # Parcourt rapidement chaque ligne du DataFrame
    tpo_raw = getattr(row, "TPO_ID", None)  # Récupère TPO_ID si présent
    color, groupe, libelle = classifier_antenne(tpo_raw)  # Détermine la couleur, le groupe et le libellé détaillé
    sup_id = getattr(row, "SUP_ID", "N/C")  # Récupère l'identifiant du support (ou 'N/C')
    com_insee = getattr(row, "COM_CD_INSEE", "N/C")  # Récupère le code INSEE de la commune (ou 'N/C')
    tpo_txt = "N/C" if pd.isna(tpo_raw) else str(int(float(tpo_raw)))  # Formate le code TPO_ID brut pour l'affichage

    popup_html = (
        f"<b>Support :</b> {sup_id}<br>"
        f"<b>Commune :</b> {com_insee}<br>"
        f"<b>Catégorie :</b> {libelle}<br>"
        f"<b>Code TPO_ID :</b> {tpo_txt}"
    )  # Contenu HTML de la fenêtre pop-up au clic sur une antenne

    # Points agrandis (radius=7) avec contour blanc bien visible sur la carte
    folium.CircleMarker(
        location=[float(row.LAT), float(row.LON)],  # Coordonnées de l'antenne
        radius=7,  # Taille du cercle
        color="#ffffff",  # Contour blanc pour ressortir sur le fond de carte
        weight=1.5,  # Épaisseur du contour
        fill=True,  # Remplit l'intérieur du cercle
        fill_color=color,  # Couleur associée à la catégorie
        fill_opacity=0.9,  # Opacité du remplissage
        popup=folium.Popup(popup_html, max_width=260),  # Associe la pop-up au point
    ).add_to(clusters_par_groupe[groupe])  # Ajoute le point dans le cluster correspondant à son groupe

  # Code HTML/CSS de la légende fixe en bas à gauche
  legende_html = """
    <div style="
        position:fixed;bottom:35px;left:20px;width:235px;
        background:#fff;z-index:9999;font-size:13px;
        border:2px solid #777;border-radius:6px;padding:10px;
        box-shadow:0 0 12px rgba(0,0,0,0.2);">
        <b>Légende - Opérateurs</b><br><br>
        <i style="background:#2563eb;width:12px;height:12px;float:left;margin-right:8px;border-radius:50%;margin-top:3px;border:1px solid #333;"></i> Bouygues Telecom<br>
        <i style="background:#ff8c00;width:12px;height:12px;float:left;margin-right:8px;border-radius:50%;margin-top:3px;border:1px solid #333;"></i> Orange<br>
        <i style="background:#e11d48;width:12px;height:12px;float:left;margin-right:8px;border-radius:50%;margin-top:3px;border:1px solid #333;"></i> SFR<br>
        <i style="background:#7c2d12;width:12px;height:12px;float:left;margin-right:8px;border-radius:50%;margin-top:3px;border:1px solid #333;"></i> Free Mobile<br>
        <i style="background:#111111;width:12px;height:12px;float:left;margin-right:8px;border-radius:50%;margin-top:3px;border:1px solid #333;"></i> Multi / Autre (détail au clic)<br>
    </div>
    """
  carte.get_root().html.add_child(folium.Element(legende_html))  # Injecte la légende dans la page HTML de la carte
  return carte  # Renvoie l'objet carte complet


def compresser_zip(fichier_html):
  """Crée une archive ZIP compressée à côté du fichier HTML."""
  fichier_zip = os.path.splitext(fichier_html)[0] + ".zip"  # Remplace l'extension .html par .zip
  with zipfile.ZipFile(
      fichier_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9
  ) as zf:  # Ouvre l'archive ZIP en écriture avec compression maximale (niveau 9)
    zf.write(fichier_html, arcname=os.path.basename(fichier_html))  # Ajoute le fichier HTML dans l'archive
  taille_mo = os.path.getsize(fichier_zip) / (1024 * 1024)  # Calcule la taille du fichier ZIP en mégaoctets (Mo)
  print(f"Archive compressée créée : {fichier_zip} ({taille_mo:.2f} Mo)")  # Affiche le chemin et la taille du ZIP


# --- EXÉCUTION PRINCIPALE ---
chemin = "C:/Users/guibeleo/Desktop/SAE15-D3/data/sup_support.txt"  # Chemin du fichier source des supports
sortie_html = "C:/Users/guibeleo/Desktop/SAE15-D3/carte_antennes_france.html"  # Chemin du fichier HTML de sortie

if os.path.exists(chemin):  # Vérifie que le fichier source existe bien
  df_antennes = charger_donnees(chemin, limite=None)  # Charge et nettoie toutes les antennes (limite=None)
  ma_carte = creer_carte(df_antennes)  # Génère la carte interactive
  ma_carte.save(sortie_html)  # Sauvegarde la carte en fichier HTML
  compresser_zip(sortie_html)  # Compresse le fichier HTML en archive ZIP
  print(f"Carte enregistrée dans : {sortie_html}")  # Confirme l'enregistrement
  display(ma_carte)  # Affiche la carte directement dans Jupyter Notebook
else:
  print(f"Erreur : Fichier introuvable à l'emplacement {chemin}")  # Message d'erreur si le fichier est absent

Chargement et nettoyage des données...
Ajout de 199290 antennes sur la carte...
